In [6]:
import numpy as np
import pandas as pd

# 1. Cargar el dataset que organizaste
df = pd.read_csv("datos_normalizados.csv") # Asegúrate que este sea el nombre

def crear_ventanas(data, window_size=60):
    X = []
    y = []
    # Usaremos todas las columnas para X
    # Y predeciremos 'ciudad_bolivar' (Asegúrate de saber su nombre exacto)
    target_col = 'ciudad_bolivar'
    target_idx = df.columns.get_loc(target_col)
    
    data_values = data.values
    for i in range(window_size, len(data_values)):
        X.append(data_values[i-window_size:i, :]) # Los 60 días previos
        y.append(data_values[i, target_idx])      # El valor a predecir
    return np.array(X), np.array(y)

# Creamos las ventanas
X, y = crear_ventanas(df, window_size=60)

# 2. División Entrenamiento / Prueba (80% - 20%)
# Como es una serie temporal, NO usamos shuffle (no mezclamos los días)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Estructura para la IA: {X_train.shape}") 
# Debería mostrar algo como: (muestras, 60, 27)

Estructura para la IA: (12808, 60, 27)


In [7]:
# En lugar de usar un número fijo, búscalo así:
target_col_name = 'ciudad_bolivar'
target_idx = df.columns.get_loc(target_col_name)

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

# 1. Cargar el original (el que tiene la columna 'fecha')
df = pd.read_csv('../../data/processed/data_ordenado.csv', parse_dates=['fecha'])

# 2. Crear ciclicidad
day_of_year = df['fecha'].dt.dayofyear
df['fecha_seno'] = np.sin(2 * np.pi * day_of_year / 365.25)
df['fecha_coseno'] = np.cos(2 * np.pi * day_of_year / 365.25)

# 3. Reordenar: Fechas al inicio, luego el resto
cols_resto = [c for c in df.columns if c not in ['fecha', 'fecha_seno', 'fecha_coseno']]
nuevo_orden = ['fecha_seno', 'fecha_coseno'] + cols_resto
df_reordenado = df[nuevo_orden]

# 4. Crear los escaladores (AQUÍ ES DONDE NACEN LOS .PKL)
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

# Ajustar X (las 27 columnas: 2 de fecha + 25 originales)
datos_x_normalizados = scaler_x.fit_transform(df_reordenado)

# Ajustar Y (solo Ciudad Bolívar)
# Usamos el nombre exacto de la columna para evitar errores
target_col = 'ciudad_bolivar'
datos_y_normalizados = scaler_y.fit_transform(df_reordenado[[target_col]])

# 5. GUARDAR LOS ARCHIVOS QUE SUBIRÁS A COLAB
# Guardar el CSV normalizado
df_final = pd.DataFrame(datos_x_normalizados, columns=nuevo_orden)
df_final.to_csv('dataset_final_colab.csv', index=False)

# GUARDAR LOS BINARIOS ACTUALIZADOS
joblib.dump(scaler_x, 'scaler_x_final.pkl')
joblib.dump(scaler_y, 'scaler_y_final.pkl')

print("✅ ¡Archivos generados con éxito!")
print(f"Sube estos 3 archivos a Colab: \n1. dataset_final_colab.csv \n2. scaler_x_final.pkl \n3. scaler_y_final.pkl")

✅ ¡Archivos generados con éxito!
Sube estos 3 archivos a Colab: 
1. dataset_final_colab.csv 
2. scaler_x_final.pkl 
3. scaler_y_final.pkl
